<a href="https://colab.research.google.com/github/eehujnihs21-stack/2555037/blob/main/2555037%EC%8B%A0%EC%A3%BC%ED%9D%AC_0403_%EC%B5%9C%EC%A2%85.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd
from scipy import stats
from sklearn.feature_extraction.text import CountVectorizer

# -------------------------------------------------------------------------
# [1단계] Books to Scrape - 상위 20개 및 1~5페이지 전체 책 제목 수집
# -------------------------------------------------------------------------
print("=== [1단계] 책 제목 수집 시작 ===")

# 세션(Session)을 재사용하여 속도 및 연결 안정성 향상
with requests.Session() as session:
    all_titles_100 = []

    for page in range(1, 6):
        url = f"https://books.toscrape.com/catalogue/page-{page}.html"
        try:
            res = session.get(url, timeout=10)
            res.raise_for_status()  # 200 OK가 아니면 에러 발생
        except requests.RequestException as e:
            print(f"페이지 {page} 로드 실패: {e}")
            continue

        soup = BeautifulSoup(res.text, 'html.parser')
        books = soup.find_all('h3')

        for book in books:
            a_tag = book.find('a')
            if a_tag and a_tag.has_attr('title'):
                all_titles_100.append(a_tag['title'])

# 데이터프레임 변환
df_titles_100 = pd.DataFrame(all_titles_100, columns=['Book Title'])

print("--- 상위 20개 책 제목 ---")
print(df_titles_100.head(20))
print(f"\n총 수집된 제목 수: {len(df_titles_100)}\n")


# -------------------------------------------------------------------------
# [2단계] 카테고리별 수집 및 가격 데이터 통계 분석 (정규성 검정)
# -------------------------------------------------------------------------
print("=== [2단계] 카테고리별 분석 및 정규성 검정 시작 ===")

categories = [
    'travel_2', 'mystery_3', 'historical-fiction_4', 'sequential-art_5',
    'classics_6', 'philosophy_7', 'romance_8', 'womens-fiction_9',
    'fiction_10', 'childrens_11'
]
base_url = "https://books.toscrape.com/catalogue/category/books/"
all_data = []

with requests.Session() as session:
    for cat in categories:
        url = f"{base_url}{cat}/index.html"
        try:
            res = session.get(url, timeout=10)
            res.raise_for_status()
        except requests.RequestException as e:
            print(f"카테고리 {cat} 로드 실패: {e}")
            continue

        soup = BeautifulSoup(res.text, 'html.parser')
        items = soup.find_all('article', class_='product_pod')[:10]

        for item in items:
            # 안전하게 데이터 추출 (에러 방지 구조)
            h3_tag = item.find('h3')
            a_tag = h3_tag.find('a') if h3_tag else None
            price_tag = item.find('p', class_='price_color')

            if a_tag and price_tag:
                title = a_tag['title']
                price_text = price_tag.text
                price = float(re.sub(r'[^\d.]', '', price_text))

                # 깔끔하게 카테고리 이름 변환
                cat_name = cat.split('_')[0].replace('-', ' ').title()

                all_data.append({
                    'Category': cat_name,
                    'Title': title,
                    'Price': price
                })

df_final = pd.DataFrame(all_data)

# 가격 기준 내림차순 정렬 후 상위 10개 출력
top_10_expensive = df_final.sort_values(by='Price', ascending=False).head(10)
print("\n--- 가장 비싼 책 Top 10 ---")
print(top_10_expensive[['Category', 'Title', 'Price']].to_string(index=False))

# 통계 분석 (정규성 검정)
shapiro_test = stats.shapiro(df_final['Price'])
skewness = df_final['Price'].skew()
kurtosis = df_final['Price'].kurtosis()

print("\n--- 통계적 유의미함 분석 결과 ---")
print(f"Shapiro-Wilk 통계량: {shapiro_test.statistic:.4f}")
print(f"p-value: {shapiro_test.pvalue:.4f}")
print(f"왜도 (Skewness): {skewness:.4f}")
print(f"첨도 (Kurtosis): {kurtosis:.4f}")

if shapiro_test.pvalue > 0.05:
    print("\n결과: p-value가 0.05보다 크므로, 귀무가설을 채택하여 '정규분포를 따른다'고 볼 수 있습니다.\n")
else:
    print("\n결과: p-value가 0.05보다 작으므로, 귀무가설을 기각하여 '정규분포를 따지 않는다'고 판단합니다.\n")


# -------------------------------------------------------------------------
# [3단계] Quotes to Scrape - 단어 빈도 분석 및 Binary Encoding
# -------------------------------------------------------------------------
print("=== [3단계] 명언 데이터 분석 및 텍스트 마이닝 시작 ===")

quotes_list = []
with requests.Session() as session:
    for page in range(1, 11):
        url = f"http://quotes.toscrape.com/page/{page}/"
        try:
            res = session.get(url, timeout=10)
            res.raise_for_status()
        except requests.RequestException as e:
            print(f"명언 페이지 {page} 로드 실패: {e}")
            continue

        soup = BeautifulSoup(res.text, 'html.parser')
        quotes = soup.find_all('span', class_='text')
        for q in quotes:
            # 특수 따옴표 및 불필요한 공백 제거
            clean_quote = q.text.strip('“”"\' ').strip()
            quotes_list.append(clean_quote)

# 1. 단어 빈도 분석 (CountVectorizer 기본형)
cv = CountVectorizer(stop_words='english')
dtm = cv.fit_transform(quotes_list)

word_counts = pd.DataFrame({
    'word': cv.get_feature_names_out(),
    'count': dtm.toarray().sum(axis=0)
})

top_5_words = word_counts.sort_values(by='count', ascending=False).head(5)
print("\n--- 가장 출현빈도가 높은 단어 Top 5 ---")
print(top_5_words.to_string(index=False))

# 2. Binary Encoding 데이터프레임 생성 (존재 여부 0 또는 1)
cv_binary = CountVectorizer(binary=True, stop_words='english')
dtm_binary = cv_binary.fit_transform(quotes_list)

df_encoding = pd.DataFrame(dtm_binary.toarray(), columns=cv_binary.get_feature_names_out())

print("\n--- Binary Encoding 데이터프레임 (상위 5행) ---")
print(df_encoding.head())
print(f"\n데이터프레임 크기: {df_encoding.shape} (행: 인용문 수, 열: 고유 단어 수)")

=== [1단계] 책 제목 수집 시작 ===
--- 상위 20개 책 제목 ---
                                           Book Title
0                                A Light in the Attic
1                                  Tipping the Velvet
2                                          Soumission
3                                       Sharp Objects
4               Sapiens: A Brief History of Humankind
5                                     The Requiem Red
6   The Dirty Little Secrets of Getting Your Dream...
7   The Coming Woman: A Novel Based on the Life of...
8   The Boys in the Boat: Nine Americans and Their...
9                                     The Black Maria
10     Starving Hearts (Triangular Trade Trilogy, #1)
11                              Shakespeare's Sonnets
12                                        Set Me Free
13  Scott Pilgrim's Precious Little Life (Scott Pi...
14                          Rip it Up and Start Again
15  Our Band Could Be Your Life: Scenes from the A...
16                                   